# TypedDict 使用

`TypedDict` 是 `typing` 模块提供的类型提示工具，用来描述字典的键和值类型。在 LangChain 中，它可以作为 `with_structured_output` 的轻量级 schema，让模型按指定结构输出，最终得到一个普通的 `dict`（而不是 Pydantic 模型实例）。

In [1]:
import os
from typing import Annotated, Literal, NotRequired, Required, TypedDict

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv(override=True)

# 与 pydantic 示例保持一致：关闭思考模式，避免结构化输出时 tool_choice 不被支持
model = init_chat_model(
    api_base=os.getenv("DEEPSEEK_API_BASE"),
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    model="deepseek-flash",
    model_provider="deepseek",
    model_kwargs={"reasoning_effort": "none"},
)


/Users/lijixu/PycharmProjects/langchain_demo/.venv/lib/python3.14/site-packages/langchain_core/utils/pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1
/Users/lijixu/PycharmProjects/langchain_demo/.venv/lib/python3.14/site-packages/langchain/chat_models/base.py:516: UserWarning: Parameters {'reasoning_effort'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  return _init_chat_model_helper(


## 基础用法

In [2]:
## 基础用法：用 TypedDict 描述结构化输出，返回的是普通 dict

class Person(TypedDict):
    """人物信息"""
    name: str          # 姓名
    age: int           # 年龄
    occupation: str    # 职位

model_person = model.with_structured_output(schema=Person)
response = model_person.invoke("小许是一个28岁的Java开发工程师")

print(response)
print("返回类型：", type(response))          # 注意这里是 dict，不是 Pydantic 模型
print("按 key 取值：", response["name"], response["age"])


{'name': '小许', 'age': 28, 'occupation': 'Java开发工程师'}
返回类型： <class 'dict'>
按 key 取值： 小许 28


## 字段描述 Annotated

In [3]:
## 字段描述：TypedDict 没有 Field()，用 Annotated 给字段附加说明，帮助模型理解语义

class Movie(TypedDict):
    """电影信息"""
    title: Annotated[str, "电影的标题"]
    director: Annotated[str, "导演"]
    year: Annotated[int, "上映年份"]
    rating: Annotated[float, "评分"]

model_movie = model.with_structured_output(schema=Movie)
print(model_movie.invoke("帮我找一下环太平洋电影的信息"))


{'title': '环太平洋'}


## 可选字段

In [4]:
## 可选字段：用 NotRequired 标记非必填字段（Python 3.11+）

class Person(TypedDict):
    """人物信息"""
    name: str
    age: NotRequired[int]          # 可以有，也可以没有
    occupation: NotRequired[str]   # 可以有，也可以没有

model_person = model.with_structured_output(schema=Person)
print(model_person.invoke("小许是一个Java开发工程师"))


TypeError: NotRequired accepts only a single type. Got (<class 'int'>,).

In [5]:
## total=False：默认所有字段都可选，再用 Required 把个别字段标记为必填

class Article(TypedDict, total=False):
    """文章信息"""
    title: Required[str]   # 必填
    author: str            # 可选
    views: int             # 可选

model_article = model.with_structured_output(schema=Article)
print(model_article.invoke("文章《LangChain 入门》作者小许，阅读量 1000"))


TypeError: Required accepts only a single type. Got (<class 'str'>,).

## 字面量 Literal

In [6]:
## 固定取值：用 Literal 限定字段只能取几个值

class Resume(TypedDict):
    """简历信息"""
    name: str
    level: Literal["初级", "中级", "高级", "专家"]   # 只能取其中之一
    city: str

model_resume = model.with_structured_output(schema=Resume)
print(model_resume.invoke("张三是高级Java开发工程师，base 在上海"))


{'name': '张三', 'level': '高级', 'city': '上海'}


## 列表字段

In [ ]:
## 列表字段：让模型一次输出多个同类型元素

class MovieList(TypedDict):
    """电影信息"""
    title: str
    actors: list[str]     # 主演列表
    genres: list[str]     # 类型标签列表

model_movie_list = model.with_structured_output(schema=MovieList)
print(model_movie_list.invoke("请介绍电影《流浪地球》的主演和类型"))


## 嵌套 TypedDict

In [7]:
## 嵌套：一个 TypedDict 可以作为另一个 TypedDict 的字段类型

class Address(TypedDict):
    """住址信息"""
    city: str       # 城市
    street: str     # 街道

class Employee(TypedDict):
    """员工信息"""
    name: str
    address: Address   # 字段类型是另一个 TypedDict

model_employee = model.with_structured_output(schema=Employee)
print(model_employee.invoke("小许住在北京市朝阳区望京街道"))


{'name': '小许', 'address': {'city': '北京市', 'street': '朝阳区望京街道'}}


## 与工具结合

In [8]:
## 与工具结合：TypedDict 也可以作为工具的 args_schema
from langchain_core.tools import tool

class WeatherInput(TypedDict):
    """天气查询参数"""
    city: Annotated[str, "城市名称，如北京、上海"]

@tool(args_schema=WeatherInput)
def get_weather(city: str) -> str:
    """查询指定城市的天气"""
    return f"{city}今天晴，25℃"

print(get_weather.invoke({"city": "北京"}))


TypeError: args_schema must be a subclass of pydantic BaseModel or a JSON schema dict. Got: <class '__main__.WeatherInput'>.

## TypedDict 与 Pydantic 的对比

- **TypedDict**：轻量、无额外依赖，`with_structured_output` 返回普通 `dict`；只在类型检查阶段生效，**运行时不做校验**，也没有默认值、别名等能力。
- **Pydantic**：功能更强，支持默认值、字段校验、别名、嵌套校验等，返回带属性的模型实例，适合需要严格校验或复杂逻辑的场景。

结论：结构简单、只想要个 `dict` 时用 `TypedDict`；需要校验和复杂能力时用 Pydantic。